In [ ]:
!pip install --upgrade openai

In [ ]:
!pip install google-generativeai anthropic

In [ ]:
from google.colab import userdata
openaiapi = userdata.get('openaiapi')
geminiapi = userdata.get('geminiapi')
claudeapi = userdata.get('claudeapi')

In [ ]:
from openai import OpenAI
from anthropic import Anthropic
import google.generativeai as genai
from IPython.display import display, Markdown, HTML




In [ ]:
openai_client = OpenAI(api_key=openaiapi, base_url = "https://openrouter.ai/api/v1")
claude_client = Anthropic(api_key=claudeapi, base_url="https://api.llmsrelay.com")
genai.configure(api_key=geminiapi)
gemini = genai.GenerativeModel("gemini-3.5-flash-lite")

In [ ]:
def print_markdown(text):
  display(Markdown(text))
def display_html_code(provider_name, html_content):
    print_markdown(f"### Generated HTML from {provider_name}:")
    display(Markdown(f"```html\n{html_content}\n```"))

In [ ]:
startup_name = "Power Gym"
startup_concept = """An intelligent gym management system that uses AI to analyze member activity, predict churn, and automate personalized engagement.
Focus on improving member retention, optimizing fitness experiences, and increasing sales efficiency for gyms of all sizes"""

In [ ]:
html_prompt = f"""
You are a helpful AI assistant acting as a front-end web developer.

Your task is to generate the complete HTML code for a simple, clean, and professional-looking landing page (index.html) for a new AI-powered gym management startup.

Startup Name: {startup_name}
Concept: {startup_concept}

Please generate ONLY the full HTML code, starting with <!DOCTYPE html> and ending with </html>.
Create a modern, visually appealing landing page with the following:

-- Don't include images in the code. Raw HTML code with inline CSS for styling.

1. A sleek header with the gym platform name in a bold, modern font and a compelling fitness-focused tagline
2. A hero section with a clear value proposition and a strong call-to-action button
3. A features section highlighting 3-4 key AI-powered benefits, such as member activity tracking, AI-powered churn prediction, personalized workout recommendations, and automated member engagement
4. A "How it Works" section with simple numbered steps explaining how gyms can use the platform
5. A testimonials section with fictional quotes from gym owners or fitness managers
6. A pricing section with at least two subscription tiers suitable for gyms of different sizes
7. A professional footer with navigation links and social media icons

Use inline CSS for styling with a modern fitness-inspired color palette using primary, secondary, and accent colors.
Include responsive design elements, subtle animations, hover effects, and whitespace for readability.

Emphasize AI capabilities, automation, ease of use, member retention, personalized fitness experiences, and business growth throughout the copy.

Focus on conversion-optimized marketing messages that address common gym challenges such as member churn, low engagement, inefficient management, and difficulty understanding member behavior.

Make the landing page feel like a real modern SaaS product rather than a generic gym website.

Do not include any explanations before or after the code block. Just provide the raw HTML code.
"""

print_markdown("**Core Prompt defined for the LLMs:**")
print_markdown(f"> {html_prompt}")

In [ ]:
openai_html_output = "OpenAI generation failed!"
try:
  response = openai_client.chat.completions.create(model = "gpt-4o-mini", messages=[{"role":"user", "content":html_prompt}])
  openai_html_output = response.choices[0].message.content
  if openai_html_output.strip().startswith("```html"):
        lines = openai_html_output.strip().splitlines()
        openai_html_output = "\n".join(lines[1:-1]).strip()
  else:
        openai_html_output = openai_html_output.strip()

  display_html_code("GPT-4O-mini", openai_html_output)
  filename = "gpt_gymsite.html"
  with open(filename, "w", encoding="utf-8") as f:
    f.write(openai_html_output)

except Exception as e:
    print_markdown(f"Error calling OpenAI API: {e}")
    openai_html_output = f"<!-- Error calling OpenAI API: {e} -->"

In [ ]:
gemini_html_output = "Failed to generate!"
try:
    response = gemini.generate_content(
        html_prompt,
    )


    raw_output = response.text
    if raw_output.strip().startswith("```html"):
        lines = raw_output.strip().splitlines()
        gemini_html_output = "\n".join(lines[1:-1]).strip()
    else:
        gemini_html_output = raw_output.strip()

    file_name = "gemini_gymsite.html"
    with open(file_name, "w", encoding="utf-8") as f:
      f.write(gemini_html_output)
      display_html_code("Gemini", gemini_html_output)

except Exception as e:
  print(e)


In [ ]:
claude_html_output = "<!-- Claude generation not run or failed -->"

print_markdown("## Calling Anthropic Claude API...")

claude_model_name = "claude-sonnet-4.6"
print_markdown(f"(Using model: {claude_model_name})")

try:
    response = claude_client.messages.create(
        model = claude_model_name,
        max_tokens = 20000,
        messages = [{"role": "user", "content": html_prompt}],
    )

    raw_output = response.content[0].text
    if raw_output.strip().startswith("```html"):
        lines = raw_output.strip().splitlines()
        claude_html_output = "\n".join(lines[1:-1]).strip()
    else:
        claude_html_output = raw_output.strip()

    display_html_code(f"Anthropic Claude ({claude_model_name})", claude_html_output)
    file_path = "claude_gymsite.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(claude_html_output)
    print_markdown(f"Successfully saved Claude output to `{file_path}`")

except Exception as e:
    print_markdown(f"Error calling Anthropic Claude API: {e}")
    claude_html_output = f"<!-- Error calling Anthropic Claude API: {e} -->"